#  Модуль 4. Корреляционный анализ

## Подробный конспект

### 4.1. Зачем изучать связи между признаками

В реальных датасетах признаки редко существуют изолированно. Рост связан с весом, площадь квартиры — с ценой, возраст — со стажем работы. **Корреляционный анализ** помогает:

- Найти признаки, которые дублируют друг друга (избыточность)
- Выявить признаки, наиболее связанные с целевой переменной
- Обнаружить скрытые закономерности в данных
- Подготовить почву для отбора признаков (Feature Selection)

> **Важно:** корреляция — это только первый шаг. Она показывает, что переменные «движутся вместе», но не объясняет, почему.

### 4.2. Корреляция ≠ Причинность

Это самая опасная ловушка для начинающих.

**Корреляция** — статистическая связь: когда одна переменная меняется, другая тоже меняется.

**Причинно-следственная связь** — когда изменение одной переменной **вызывает** изменение другой.

#### Классические примеры ложной корреляции

| Переменная X | Переменная Y | Корреляция | Реальная причина |
|-------------|-------------|-----------|-----------------|
| Продажи мороженого | Число утоплений | Высокая | Жара (третья переменная) |
| Число пиратов | Глобальное потепление | Высокая | Случайность / третий фактор (время) |
| Импорт лимонов из Мексики | Число ДТП | Высокая | Рост экономики обоих показателей |

**Вывод:** если X коррелирует с Y, это не значит, что X -> Y. Возможны три объяснения:
1. **X вызывает Y** (причинность)
2. **Y вызывает X** (обратная причинность)
3. **Z вызывает и X, и Y** (скрытая переменная — конфаундер)

В ML это важно: мы используем корреляцию для **предсказания**, а не для объяснения механизмов.

### 4.3. Коэффициент корреляции Пирсона (Pearson)

Самый известный коэффициент. Измеряет **силу линейной связи** между двумя числовыми переменными.

#### Что такое «линейная связь»

Если нарисовать точки на графике и они выстраиваются примерно в линию — связь линейная.

In [ ]:
Сильная положительная:     Слабая положительная:     Отрицательная:
Y ↑  ●                     Y ↑    ●  ●               Y ↑ ●
  │    ●  ●                  │  ●        ●             │   ●
  │      ●  ●                │●    ●  ●                │     ●
  └──────-> X                 └──────-> X                └──────-> X

#### Формула (для понимания)

$$r_{XY} = \frac{\sum_{i=1}^{n}(X_i - \bar{X})(Y_i - \bar{Y})}{\sqrt{\sum_{i=1}^{n}(X_i - \bar{X})^2 \cdot \sum_{i=1}^{n}(Y_i - \bar{Y})^2}}$$

**Что происходит в числителе:** если X выше своего среднего и Y тоже выше своего среднего — произведение положительное. Если одно выше, а другое ниже — отрицательное. Если связи нет — положительные и отрицательные произведения сокращаются.

**Диапазон значений:**
- **+1** — идеальная прямая линейная связь (чем больше X, тем больше Y)
- **0** — линейной связи нет
- **−1** — идеальная обратная линейная связь (чем больше X, тем меньше Y)

#### Интерпретация силы связи (общепринятая шкала)

| \|r\| | Интерпретация |
|-------|--------------|
| 0.00 – 0.19 | Очень слабая |
| 0.20 – 0.39 | Слабая |
| 0.40 – 0.59 | Умеренная |
| 0.60 – 0.79 | Сильная |
| 0.80 – 1.00 | Очень сильная |

#### Предпосылки Пирсона (критически важно!)

Чтобы коэффициент Пирсона имел смысл, нужно:

1. **Линейность** — связь должна быть похожа на прямую линию
2. **Нормальность** — обе переменные желательно нормально распределены
3. **Отсутствие выбросов** — один выброс может сильно исказить r
4. **Непрерывность** — обе переменные числовые

#### Пример в Python

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Данные с линейной связью
np.random.seed(42)
x = np.random.normal(50, 10, 100)
y = 2 * x + np.random.normal(0, 5, 100)  # y линейно зависит от x + шум

# Метод 1: pandas
df = pd.DataFrame({'x': x, 'y': y})
print(df['x'].corr(df['y']))  # ≈ 0.97 — сильная положительная

# Метод 2: scipy (даёт ещё и p-value)
r, p_value = pearsonr(x, y)
print(f"r = {r:.3f}, p-value = {p_value:.2e}")
# p-value < 0.05 — корреляция статистически значима

#### Когда Пирсон обманывает

In [ ]:
# Пример 1: Парабола (нелинейная связь)
x = np.linspace(-10, 10, 100)
y = x ** 2  # идеальная квадратичная зависимость

r, _ = pearsonr(x, y)
print(r)  # ≈ 0.0 — Пирсон говорит "связи нет", но она есть!

# Пример 2: Один выброс
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 100])
y = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 100])

r, _ = pearsonr(x, y)
print(r)  # ≈ 1.0 — "идеальная связь" из-за одной точки, хотя остальные слабо связаны

### 4.4. Коэффициент корреляции Спирмена (Spearman)

Измеряет **силу монотонной связи** через ранги.

#### Что такое «ранги»

Вместо исходных значений мы используем их **порядковые номера** после сортировки.

In [ ]:
# Исходные данные
зарплата = [30000, 50000, 25000, 80000, 45000]

# Ранги (1 = минимум)
# 25000 -> 1, 30000 -> 2, 45000 -> 3, 50000 -> 4, 80000 -> 5
ранг_зарплаты = [2, 4, 1, 5, 3]

Спирмен считает корреляцию Пирсона, но уже между рангами.

#### Когда использовать Спирмена

- Есть **выбросы** (ранги устойчивы к ним)
- Связь **монотонная**, но не линейная (логарифм, экспонента)
- Данные **порядковые** (шкала «плохо — средне — хорошо»)
- Распределение **далеко от нормального**

In [ ]:
from scipy.stats import spearmanr

# Нелинейная монотонная связь
x = np.arange(1, 100)
y = np.log(x)  # логарифмическая связь

# Пирсон увидит связь, но не идеальную
r_pearson, _ = pearsonr(x, y)

# Спирмен увидит почти идеальную монотонную связь
r_spearman, _ = spearmanr(x, y)

print(f"Пирсон: {r_pearson:.3f}")    # ≈ 0.85
print(f"Спирмен: {r_spearman:.3f}")  # ≈ 1.0

#### Пример с выбросами

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 1000])
y = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 1000])

print(pearsonr(x, y)[0])    # ≈ 1.0 (выброс доминирует)
print(spearmanr(x, y)[0])   # ≈ 1.0 (здесь тоже 1.0, но попробуйте изменить y выброса)

y[9] = 20  # сделаем выброс нелогичным
print(pearsonr(x, y)[0])    # упадёт сильно
print(spearmanr(x, y)[0])   # останется высоким, т.к. ранги сохраняют порядок

### 4.5. Коэффициент корреляции Кендалла (Kendall)

Также основан на рангах, но считает **согласованность пар** (concordant/discordant pairs).

#### Как работает (упрощённо)

Берём все возможные пары наблюдений. Если при росте X растёт Y — пара **согласованная** (concordant). Если падает — **несогласованная** (discordant).

$$\tau = \frac{(\text{число согласованных пар}) - (\text{число несогласованных пар})}{\text{всего пар}}$$

#### Когда использовать Кендалла

- **Маленькие выборки** (n < 30)
- **Много совпадающих рангов** (ties) — одинаковых значений
- **Порядковые данные** с небольшим числом градаций
- Когда важна **интерпретация**: τ — это вероятность того, что две случайные пары согласованы

In [ ]:
from scipy.stats import kendalltau

# Пример с категориальными порядковыми данными
образование = [1, 2, 2, 3, 3, 3, 4, 4, 5]  # 1=начальное, 5=высшее
доход = [2, 2, 3, 3, 4, 4, 4, 5, 5]

tau, p = kendalltau(образование, доход)
print(f"Кендалл τ = {tau:.3f}")  # ≈ 0.87 — сильная согласованность

### 4.6. Сравнение трёх коэффициентов

| Критерий | Пирсон | Спирмен | Кендалл |
|----------|--------|---------|---------|
| **Что измеряет** | Линейную связь | Монотонную связь рангов | Согласованность пар |
| **Чувствительность к выбросам** | Высокая | Низкая | Низкая |
| **Требует нормальности** | Да | Нет | Нет |
| **Работает с порядковыми данными** | Нет | Да | Да |
| **Устойчивость к совпадениям рангов** | — | Средняя | Высокая |
| **Скорость расчёта** | Быстро | Быстро | Медленно (большие n) |
| **Диапазон** | [−1, 1] | [−1, 1] | [−1, 1] |
| **Интерпретация** | Линейная зависимость | Монотонная зависимость | Вероятность согласованности пар |

**Практическое правило:**
- Данные нормальны, линейны, без выбросов -> **Пирсон**
- Есть выбросы или нелинейная монотонная связь -> **Спирмен**
- Маленькая выборка, много одинаковых значений -> **Кендалл**

### 4.7. Матрица корреляций

Когда признаков много, смотреть пары по отдельности неудобно. Строим **матрицу**, где на пересечении строки i и столбца j стоит корреляция между признаками i и j.

#### Построение в pandas

In [ ]:
# По умолчанию — Пирсон
corr_matrix = df.corr()

# Спирмен
corr_spearman = df.corr(method='spearman')

# Кендалл
corr_kendall = df.corr(method='kendall')

#### Визуализация (повторение и углубление)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Выбираем только числовые столбцы
num_df = df.select_dtypes(include=[np.number])

# Считаем корреляцию Пирсона
corr = num_df.corr()

# Создаём маску для верхнего треугольника (матрица симметрична)
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(corr,
            mask=mask,              # скрываем верхний треугольник
            annot=True,             # числа в ячейках
            fmt='.2f',              # 2 знака после запятой
            cmap='RdBu_r',          # красный-синий (перевёрнутый)
            center=0,               # центр шкалы — белый цвет
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})

plt.title('Матрица корреляций Пирсона (нижний треугольник)')
plt.show()

#### Автоматический поиск сильно коррелирующих пар

In [ ]:
def get_high_correlations(df, threshold=0.8):
    """Находит пары признаков с корреляцией выше threshold"""
    corr = df.corr().abs()  # берём модули
    
    # Создаём маску: верхний треугольник + диагональ
    mask = np.triu(np.ones_like(corr, dtype=bool))
    corr = corr.mask(mask)
    
    # Находим пары выше порога
    high_corr = (corr > threshold).stack().reset_index()
    high_corr.columns = ['Признак_1', 'Признак_2', 'Корреляция']
    
    return high_corr.sort_values('Корреляция', ascending=False)

# Использование
high_pairs = get_high_correlations(num_df, threshold=0.8)
print(high_pairs)

### 4.8. Мультиколлинеарность

#### Что это

**Мультиколлинеарность** — ситуация, когда два или более признака сильно коррелируют **между собой** (не с таргетом!).

Пример:
- `площадь_квартиры_м2` и `площадь_квартиры_футы` — один признак в разных единицах
- `возраст` и `стаж_работы` — часто идут рука об руку
- `количество_комнат` и `площадь` — тесно связаны

#### Почему это опасно для ML

1. **Линейные модели** (линейная регрессия, логистическая регрессия):
   - Коэффициенты становятся нестабильными (малое изменение данных -> большое изменение весов)
   - Сложно интерпретировать влияние каждого признака отдельно
   - Стандартные ошибки коэффициентов завышаются

2. **Деревья и бустинги**:
   - Менее чувствительны, но избыточные признаки замедляют обучение
   - Модель может случайно выбрать один из дублей, а не лучший

#### Диагностика 1: матрица корреляций

Простое правило: если |корреляция| между двумя признаками > 0.7–0.9 — есть мультиколлинеарность.

In [ ]:
# Находим пары с корреляцией > 0.85
high_pairs = get_high_correlations(num_df, threshold=0.85)
print(high_pairs)

#### Диагностика 2: VIF (Variance Inflation Factor)

Более продвинутый метод. VIF показывает, **во сколько раз увеличивается дисперсия коэффициента** из-за наличия других признаков.

$$VIF_i = \frac{1}{1 - R_i^2}$$

Где $R_i^2$ — коэффициент детерминации из регрессии признака i на все остальные признаки.

**Интерпретация VIF:**

| VIF | Интерпретация |
|-----|--------------|
| 1 | Признак не коррелирует с другими (идеально) |
| 1–5 | Умеренная корреляция (обычно приемлемо) |
| 5–10 | Сильная корреляция (возможна проблема) |
| > 10 | Очень сильная мультиколлинеарность (требуется действие) |

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(df):
    """Рассчитывает VIF для всех числовых признаков"""
    # Убираем пропуски
    df_clean = df.dropna()
    
    # VIF требует, чтобы не было константных столбцов
    df_clean = df_clean.loc[:, df_clean.nunique() > 1]
    
    vif_data = pd.DataFrame()
    vif_data['Признак'] = df_clean.columns
    vif_data['VIF'] = [variance_inflation_factor(df_clean.values, i) 
                       for i in range(df_clean.shape[1])]
    
    return vif_data.sort_values('VIF', ascending=False)

# Использование
vif_results = calculate_vif(num_df)
print(vif_results)

**Что делать с мультиколлинеарностью:**

| Стратегия | Когда применять |
|-----------|----------------|
| **Удалить один из пары** | Если признаки дублируют друг друга (м² и футы) |
| **Объединить в один** | Среднее, взвешенная сумма, PCA |
| **Оставить оба** | Если деревья/бустинги и признаки семантически разные |
| **Регуляризация** | Lasso/Ridge сама «задавит» лишние признаки |

### 4.9. Корреляция с целевой переменной

Один из способов отбора признаков: оставить те, что сильнее всего связаны с целевой переменной (target).

In [ ]:
# Предположим, 'цена' — это наш target
target = 'цена'
features = [col for col in num_df.columns if col != target]

# Считаем корреляцию каждого признака с таргетом
target_corr = num_df[features].corrwith(num_df[target]).sort_values(key=abs, ascending=False)

print("Корреляция с целевой переменной:")
print(target_corr)

# Визуализация
plt.figure(figsize=(8, 6))
target_corr.plot(kind='barh', color=['green' if x > 0 else 'red' for x in target_corr])
plt.title('Корреляция признаков с целевой переменной')
plt.xlabel('Коэффициент корреляции Пирсона')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.show()

#### Ловушки при отборе по корреляции

1. **Нелинейная связь:** признак квадратично связан с таргетом, но Пирсон покажет 0.
2. **Взаимодействие признаков:** `признак_1` и `признак_2` по отдельности бесполезны, но вместе очень информативны.
3. **Категориальные признаки:** Пирсон не работает с категориями напрямую (нужно кодировать или использовать ANOVA/Mutual Information).

#### Альтернатива: Mutual Information (взаимная информация)

Измеряет **любую** статистическую зависимость (не только линейную). Полезен для нелинейных связей.

In [ ]:
from sklearn.feature_selection import mutual_info_regression

X = num_df.drop('цена', axis=1)
y = num_df['цена']

# Рассчитываем mutual information
mi = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)

print("Mutual Information с целевой переменной:")
print(mi_series)

### 4.10. Практический пример: полный корреляционный анализ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, kendalltau
from statsmodels.stats.outliers_influence import variance_inflation_factor

np.random.seed(42)

# Создаём реалистичные данные о квартирах
n = 500
df = pd.DataFrame({
    'площадь_м2': np.random.normal(55, 15, n).clip(20, 150),
    'этаж': np.random.randint(1, 25, n),
    'год_постройки': np.random.randint(1960, 2024, n),
    'расстояние_до_метро_м': np.random.exponential(800, n).clip(100, 5000),
})

# Создадим мультиколлинеарность: площадь в футах
df['площадь_футов'] = df['площадь_м2'] * 10.764

# Создадим целевую переменную (цену) с линейной зависимостью + шум
df['цена'] = (df['площадь_м2'] * 150000 + 
              (2024 - df['год_постройки']) * (-2000) + 
              df['расстояние_до_метро_м'] * (-30) + 
              np.random.normal(0, 500000, n))

# Добавим нелинейный признак
df['возраст_дома'] = 2024 - df['год_постройки']
df['возраст_дома_лог'] = np.log1p(df['возраст_дома'])

print("=" * 60)
print("ШАГ 1: МАТРИЦА КОРРЕЛЯЦИЙ")
print("=" * 60)

num_cols = ['площадь_м2', 'площадь_футов', 'этаж', 'год_постройки', 
            'расстояние_до_метро_м', 'возраст_дома', 'цена']
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Матрица корреляций')
plt.show()

print("\n" + "=" * 60)
print("ШАГ 2: ПОИСК СИЛЬНО КОРРЕЛИРУЮЩИХ ПАР")
print("=" * 60)

# Пары с |r| > 0.9
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.9:
            high_corr.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))

for pair in high_corr:
    print(f"{pair[0]} <-> {pair[1]}: r = {pair[2]:.3f}")

print("\n" + "=" * 60)
print("ШАГ 3: VIF (МУЛЬТИКОЛЛИНЕАРНОСТЬ)")
print("=" * 60)

vif_df = df[['площадь_м2', 'площадь_футов', 'этаж', 'год_постройки', 
             'расстояние_до_метро_м', 'возраст_дома']].dropna()

vif_data = pd.DataFrame()
vif_data['Признак'] = vif_df.columns
vif_data['VIF'] = [variance_inflation_factor(vif_df.values, i) 
                   for i in range(vif_df.shape[1])]
print(vif_data.sort_values('VIF', ascending=False))

print("\n" + "=" * 60)
print("ШАГ 4: КОРРЕЛЯЦИЯ С ЦЕЛЕВОЙ ПЕРЕМЕННОЙ")
print("=" * 60)

target_corr = df[num_cols].drop('цена', axis=1).corrwith(df['цена']).sort_values(key=abs, ascending=False)
print(target_corr)

print("\n" + "=" * 60)
print("ШАГ 5: СРАВНЕНИЕ ПИРСОНА И СПИРМЕНА")
print("=" * 60)

# Для нелинейного признака: возраст дома vs цена
r_pearson, _ = pearsonr(df['возраст_дома'], df['цена'])
r_spearman, _ = spearmanr(df['возраст_дома'], df['цена'])

print(f"Возраст дома vs Цена:")
print(f"  Пирсон:  {r_pearson:.3f}")
print(f"  Спирмен: {r_spearman:.3f}")
print(f"  Спирмен сильнее -> связь нелинейная (цена падает, но не пропорционально)")

### 4.11. Чек-лист для самопроверки

Перед переходом к Модулю 5 убедитесь, что вы:

- [ ] Понимаете разницу между корреляцией и причинностью
- [ ] Можете привести пример ложной корреляции
- [ ] Знаете, что измеряет коэффициент Пирсона и его предпосылки
- [ ] Понимаете, почему Пирсон «не видит» параболическую связь
- [ ] Знаете, в чём суть ранговой корреляции Спирмена
- [ ] Можете отличить ситуации для Пирсона, Спирмена и Кендалла
- [ ] Умеете строить матрицу корреляций и маскировать верхний треугольник
- [ ] Можете программно найти пары признаков с высокой корреляцией
- [ ] Понимаете, что такое мультиколлинеарность и почему она опасна
- [ ] Умеете считать VIF и интерпретировать его значения
- [ ] Можете оценить корреляцию признаков с целевой переменной
- [ ] Знаете, что Mutual Information — альтернатива для нелинейных связей

> **Переход к Модулю 5:** Теперь, когда вы умеете находить связи между признаками, пора научиться устранять «грязь» в данных. Мы займёмся очисткой: пропусками, дубликатами, некорректными значениями и несогласованностью форматов.